# 04 — The per-night table (handoff §1b)

    weather_hours         = |W ∩ N|            W = WEATHER bit (from 2026-02-25) or EFD criteria (before)
    on_sky_hours          = |S ∩ N|            S = visit coverage, gaps > gap_min are loss
    nonweather_loss_hours = |N| − |W ∪ S|      fault + idle, reported as "fault" for the talks (Bruno)

Then the cloud step decided by Bruno on 2026-10-18 (option 2): before the WEATHER bit, `closed_unexplained_hours` (dome shut
or telescope at horizon with covers on, no EFD criterion met) moves into `weather_hours` **only** on nights where the night
report (Source 2) gives P(weather) ≥ 0.5. Those nights carry `weather_source = "EFD criteria + night report (cloud)"` and the
moved hours in `cloud_reassigned_hours`. The `label` is the handoff's presentation rule, not a definition.

Output: `data/processed/per_night.csv` + `.parquet`; `per_night_v0.csv` is the table before the cloud step.

In [ ]:
import sys, logging
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, "..")
from time_loss import DAY_OBS_MIN, DAY_OBS_MAX, GAP_MIN_MINUTES, GAP_MIN_SENSITIVITY, DAY_OBS_WEATHER_BIT_START
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 300); pd.set_option("display.max_colwidth", 80)
logging.basicConfig(level=logging.WARNING)
for name in ("httpx", "numexpr", "time_loss.access"):
    logging.getLogger(name).setLevel(logging.WARNING)
STEP0 = Path("../data/step0"); CACHE = Path("../data/cache"); PROC = Path("../data/processed")
for p in (STEP0, CACHE, PROC): p.mkdir(parents=True, exist_ok=True)

In [ ]:
import pickle
from time_loss import accounting as A
limits = pd.read_csv(STEP0 / "night_limits.csv")
wx = pd.read_csv(STEP0 / "weather_windows_per_night.csv")
with open(CACHE / "weather_windows.pkl", "rb") as fh: ww = pickle.load(fh)
with open(CACHE / "visit_coverage.pkl", "rb") as fh: cov = pickle.load(fh)
avail = dict(zip(wx.day_obs, wx.bit_available))
wwin = {d: (ww["bit"][d] if avail[d] else ww["efd"][d]) for d in limits.day_obs}
wsrc = dict(zip(wx.day_obs, wx.weather_source))
v0 = A.per_night_table(limits, cov, wwin, wsrc, gap_min=GAP_MIN_MINUTES)
v0 = v0.merge(wx[["day_obs", "bit_weather_hours", "efd_weather_hours", "efd_wind_hours", "efd_dewpoint_hours", "efd_dynalene_hours", "closed_unexplained_hours"]], on="day_obs")
v0.to_csv(PROC / "per_night_v0.csv", index=False)
v0[["night_hours_18", "night_hours_12", "weather_hours", "on_sky_hours", "nonweather_loss_hours", "weather_and_onsky_hours"]].sum().round(1)

In [ ]:
s2 = pd.read_csv(STEP0 / "nightlog_source2_per_night.csv")
t = A.reassign_unexplained_closures(v0, s2)
t.to_csv(PROC / "per_night.csv", index=False); t.to_parquet(PROC / "per_night.parquet")
print("cloud reassignment:", (t.cloud_reassigned_hours > 0).sum(), "nights,", round(t.cloud_reassigned_hours.sum(), 1), "h")
print(t[["night_hours_18", "weather_hours", "on_sky_hours", "nonweather_loss_hours"]].sum().round(1).to_dict())
print(t.label.value_counts().to_dict())
t.groupby("weather_source")[["night_hours_18", "weather_hours", "on_sky_hours", "nonweather_loss_hours"]].agg(["sum", "count"]).round(1)

## Monthly summary

In [ ]:
t["month"] = t.day_obs // 100
m = t.groupby("month")[["night_hours_18", "weather_hours", "on_sky_hours", "nonweather_loss_hours"]].sum().round(1)
m["weather_frac"] = (m.weather_hours / m.night_hours_18).round(3); m["loss_frac"] = (m.nonweather_loss_hours / m.night_hours_18).round(3); m["onsky_frac"] = (m.on_sky_hours / m.night_hours_18).round(3)
m.to_csv(PROC / "per_month.csv"); m

## The full table, for spot checks against the human tables

In [ ]:
cols = ["day_obs", "night_hours_18", "weather_hours", "weather_source", "cloud_reassigned_hours", "on_sky_hours", "nonweather_loss_hours", "label", "weather_windows_utc"]
t[cols]